In [1]:
# Imports (all in one cell)
import os
import time
from pathlib import Path
from collections import defaultdict
import pandas as pd
from bs4 import BeautifulSoup

# Import helper functions from lib
from lib.clean_html_utils import (
    build_selector,
    load_and_cache_html_soups,
    audit_selectors,
    remove_elements_and_save,
    lint_and_format_html,
    preview_removals,
    risk_summary,
    count_elements_per_file,
    # Defensive one-by-one review
    generate_review_queue,
    review_progress,
    interactive_review,
    apply_queue_decisions,
    reset_review_queue,
)

# HTML Cleaning Workflow: Smart Element Removal for NKG

## 🎯 The Problem You're Solving

Your HR SaaS platform has a lot of **unnecessary UI chrome** (mobile menus, loading bars, chat windows, modals) cluttering the HTML. These add noise to your Navigational Knowledge Graph (NKG). You want to remove them safely — but you're worried about accidentally deleting something navigation-critical (like a page-specific button or form).

The unequal line-removal you saw means some pages have more of this chrome than others. That's normal, but you need **confidence** that you're not removing anything that affects navigation.

---

## 📋 The Workflow (3 Steps)

### **Step 0: Set Up** (Cells 2–4)
- **Cell 2:** Load the HTML files from `data/scraped_html/`
- **Cell 3:** Define your `CANDIDATES` — the CSS selectors for elements you *suspect* are pure chrome (e.g., `div.menu-mobile`, `div#loading-global`)
- **Cell 4 (⚡ NEW):** **Pre-cache HTML files** — parse all files once upfront, then reuse the cache. **This makes Steps 1–1.6 run 15–20x faster!**

### **Step 1: Audit & Risk Assessment** (Cells 5–10)
Three sub-steps, run in order:

#### **Step 1 (Cell 6):** Global Audit
Checks: "*Which selectors actually exist in my files?*"
- Shows you `coverage` = how many files contain each selector
- Helps you spot page-specific vs. global chrome

#### **Step 1.5 (Cell 8):** Risk Assessment ⚠️
**This is the key decision point.**

For each selector, the system asks: "*Does the element I'm about to remove contain any interactive children?*"
- **SAFE** = No interactive descendants (pure layout/divs) → **auto-approved for removal**
- **RISKY** = Contains buttons, links, inputs, etc. → **needs your explicit approval**

Output: `df_risk` table showing risk per selector + `df_preview` showing exactly which elements are RISKY.

**👉 YOU DECIDE HERE:** Look at the RISKY elements. Ask yourself: "Is this interactive content global chrome that appears on every page (like a hamburger menu duplicated in the sidebar)?" If yes, you'll approve it in Step 2a. If no (page-specific action), you'll leave it blocked.

#### **Step 1.6 (Cell 10):** Variance Diagnostic
Answers: "*Why are some files losing more lines than others?*"
- Shows a pivot table: rows = files, columns = selectors, value = how many elements matched
- High `TOTAL` column? That file has more chrome candidates
- Zero in a cell? That selector doesn't exist in that file (expected)

**Why this matters:** If a "global" selector like `div.menu-mobile` shows 0 matches on some pages, either:
  - It's actually page-specific (suspicious)
  - Or the page doesn't have mobile nav (also okay)

---

### **Step 2: Controlled Removal** (Cells 11–13)

#### **Step 2a (Cell 12):** Approval Gate
You fill in the `OVERRIDE_ALLOW` dictionary:
```python
OVERRIDE_ALLOW = {
    "Mobile nav menu": True,    # ✅ Approve removal
    "Side Bar Top": False,      # ❌ Keep it (too risky)
}
```

**Rules:**
- `SAFE` selectors = automatically approved, skip them in the dict
- `RISKY` selectors = you must add them and set `True` (approved) or `False` (blocked)
- If a `RISKY` selector is not in the dict, it's blocked by default

After editing, the cell prints:
```
✓ Auto-approved (SAFE):
    Audio notif
    ...

✓ Override-approved (RISKY):
    Mobile nav menu
    
✗ Blocked (RISKY, no override):
    Some element
```

#### **Step 2b (Cell 13):** Execute Removal
Removes all `approved` selectors from `data/scraped_html/` and saves to `data/cleaned_html/`.
- Uses `scraped_html/` as source (not currently cleaned files, so you can re-run)
- Shows `total_removed` per file — variance expected due to page-specific content
- Warns if any `blocked` selectors exist (remind you to decide)

---

### **Step 3: Format** (Cells 14–15)
Prettifies all cleaned HTML (indentation, readability). Optional but recommended.

---

## 🚀 How to Use This (Quick Start)

1. **Run Cell 1** (imports)
2. **Run Cell 2** (load files)
3. **Run Cell 3** (define CANDIDATES) — edit this if you want different selectors
4. **Run Cell 4** ⚡ **(PRE-CACHE HTML)** — do this once before Steps 1 & 1.5!
5. **Run Cell 6** (global audit) — understand coverage (~2 seconds!)
6. **Run Cell 8** (risk assessment) ← **LOOK AT THE OUTPUT HERE** (~3 seconds!)
   - Scan `df_risk` for RISKY selectors
   - Check `df_preview` (RISKY only) — is this chrome you want gone?
7. **Run Cell 10** (variance diagnostic) — confirm variance makes sense (~2 seconds!)
8. **Edit Step 2a** (OVERRIDE_ALLOW) — add your decisions
9. **Run Cell 12** (approval summary) — verify your choices
10. **Run Cell 13** (removal) — clean the HTML
11. **Run Cells 14–15** (format) — prettify

---

## 💡 Key Insight

You're not choosing whether to remove elements blindly. You're:
1. **Automatically approving** pure chrome (no interactive children)
2. **Explicitly reviewing** risky stuff (has buttons/links)
3. **Documenting your decisions** (via OVERRIDE_ALLOW comments)

If you accidentally approve something and it breaks your NKG, you can:
- Revert the override in Step 2a
- Re-run Step 2b (it reads from `scraped_html/` again)
- Try again

---

## ⚠️ Common Mistakes

- **Mistake:** Skipping Cell 4 (pre-cache) and wondering why Step 1 is slow
  - **Fix:** Always run the pre-cache cell first!

- **Mistake:** Approving a RISKY selector without reading `df_preview`
  - **Fix:** Always check the detail table for labels you don't recognize
  
- **Mistake:** Not running Step 1.5 & 1.6, just guessing at CANDIDATES
  - **Fix:** Let the data guide you — audit first, decide second

- **Mistake:** Forgetting to add entries to OVERRIDE_ALLOW for RISKY items you want removed
  - **Fix:** The cell prints which selectors are blocked; add them explicitly if needed

---

## 🎓 Next Steps After Cleaning

Once you're happy with the cleaned HTML:
- Feed `data/cleaned_html/` into your NKG builder
- Verify the NKG has the right Pages and Elements
- Test with your agent queries

In [23]:
# Set up data directory and list HTML files
data_dir  = Path("../data/cleaned_html_2")
html_files = sorted(data_dir.glob("*.html"))
print(f"Found {len(html_files)} HTML files")

Found 47 HTML files


In [ ]:
# Candidate selectors to audit — add/remove entries as needed
# You can use simple selectors or parent-child combos:
CANDIDATES = {
    "Mobile nav menu (sidebar/navbar mobile)"  : build_selector("div.menu-mobile"),
    "Desktop side menu (sidebar/navbar desktop)": build_selector("div.menu-w"), # dashboard, customer_organization, quick guide punya versi sendiri
    "Top header bar"   : build_selector("div.top-bar"),
    "Loading bar"      : build_selector("div#progress-bar-loading"), # progress bar antrian hitung ulang. kadang kadang aja muncul.
    "Onboarding modal" : build_selector("div.onboarding-modal"), # pengumuman "Yuk Lihat Informasi Terbaru Fingerspot.io. dibawahnya ada surat gitu" tapi ada yang pilih salah satu gitu. tambah izin karyawan
    "Audio notif"      : build_selector("audio#sound_notif"),

    # SELECT queue mode (recommended): review <select>, not thousands of <option>.
    # If you mark REMOVE in interactive queue, helper now trims option #2..N and keeps option #1.
    "Select dropdowns (keep 1 option sample)": "select",
}

# add more
CANDIDATES.update({
    "Search Bar"    :   "div.mobile-search-header",
    "Side Bar Top"      :   "div.menu-and-user",
    "Side Bar Bottom"   :   "div.menu-w",
    "Chat"              : "div.sb-main",
    "Recalculate Queue" : "div#recalculate_queue",
    "Recalculate Queue V2" : "div#recalculate_queue_progress_v2",
    "Modal Info IOS"    : "div#modal_info_ios",
    "Weird Igtrial Watermark": "div#__ig_wm__",
    "Loading Global"    :   "div.loading-global",
    "Weird Display Type" :   "div.display-type"
})

# CANDIDATES = {
#     "Ketentuan Pengguna IOS": build_selector("div#modal_info_ios")
# }

## Notes:

1. "Antrian Hitung Ulang", ini mau ditaruh di mana? Global elements diapain di NKG nya?

## ⚡ Pre-Cache: Load & Parse All HTML Once

The slowdown in Step 1 comes from **re-parsing each HTML file for every selector check**.  
By pre-parsing all files upfront and caching the BeautifulSoup objects, you save ~95% of parsing time.

**Run this cell once,** before Step 1. Then reuse `soup_cache` in all subsequent cells (Step 1.5 & 1.6) — they'll finish in seconds instead of minutes.

In [33]:
# Pre-cache all HTML files (do this once, reuse the cache)
import time
start = time.time()
soup_cache = load_and_cache_html_soups(html_files)
elapsed = time.time() - start
print(f"✓ Cached {len(soup_cache)}/{len(html_files)} HTML files in {elapsed:.2f}s")
print(f"  Memory usage: ~{sum(len(str(s)) for s in soup_cache.values()) / 1_000_000:.1f} MB")

✓ Cached 47/47 HTML files in 3.92s
  Memory usage: ~2.1 MB


## Step 1 — Audit: verify which elements exist across all pages

Each candidate selector is checked against every HTML file.  
`count` = number of files that contain at least one matching element.  
`total` = total number of scraped files.  
Proceed to removal only for selectors with `count == total` (or intentionally partial).

In [34]:
# Step 1 — Audit: verify which elements exist across all pages
start = time.time()
df_audit = audit_selectors(html_files, CANDIDATES, soup_cache=soup_cache)
elapsed = time.time() - start
print(f"Audit completed in {elapsed:.2f}s\n")
print(df_audit[["label", "selector", "coverage", "all_pages"]].to_string(index=False))

Audit completed in 0.04s

                                  label selector coverage  all_pages
Select dropdowns (keep 1 option sample)   select    37/47      False


## Step 1.5 — Risk Assessment: flag selectors that contain interactive children

For each selector, every matched element is inspected for **NKG-relevant descendants**:
`<a>`, `<button>`, `<input>`, `<select>`, `<textarea>`.

| Risk | Meaning | Action |
|------|---------|--------|
| **SAFE** | No interactive children — pure chrome/layout noise | Auto-approved for removal |
| **RISKY** | Contains interactive descendants — may carry navigational signal | Requires explicit `OVERRIDE_ALLOW` entry before removal |

Review the detail table (`df_preview`) and the summary (`df_risk`) to decide on overrides.

In [19]:
# # Step 1.5 — Risk Assessment
# start = time.time()
# df_preview = preview_removals(html_files, CANDIDATES, soup_cache=soup_cache)
# df_risk    = risk_summary(df_preview)
# elapsed = time.time() - start
# print(f"Risk assessment completed in {elapsed:.2f}s\n")

# print("=== Risk Summary (one row per selector) ===")
# display(df_risk.sort_values("risk", ascending=False))

# print("\n=== Detailed Preview (RISKY elements only) ===")
# risky_detail = df_preview[df_preview["risk"] == "RISKY"][
#     ["label", "file", "el_index", "tag", "id", "classes",
#      "interactive_children", "interactive_tags", "text_preview"]
# ]
# display(risky_detail if not risky_detail.empty else "No RISKY elements found — all selectors are SAFE.")

## Step 1.6 — Variance Diagnostic: why is line-removal unequal across files?

The pivot table below shows **how many elements** each selector matches per file.  
Files with a high `TOTAL` column are the ones losing the most lines — you can trace exactly which selector(s) are responsible.  
Zero in a cell means that selector has no match in that file (expected for page-specific elements).

> **Key insight:** variance is *expected* for page-specific content.  
> Only investigate when a selector that should be global chrome shows a non-uniform count.

In [20]:
# # Step 1.6 — Per-file element count pivot (variance diagnostic)
# # Uses the *scraped* HTML files so results are not affected by prior cleaning runs.
# start = time.time()
# source_files = sorted(Path("../data/scraped_html").glob("*.html"))
# source_cache = load_and_cache_html_soups(source_files)
# df_variance = count_elements_per_file(source_files, CANDIDATES, soup_cache=source_cache)
# elapsed = time.time() - start
# print(f"Variance diagnostic completed in {elapsed:.2f}s\n")

# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", 200)
# display(df_variance)

---
## Step 2 — Defensive One-by-One Review

Review every matched element **individually** before anything gets deleted.

### How it works
1. **Generate queue** — scans all files, creates `data/review_queue.json` with one item per matched element. If the file already exists, your previous decisions are preserved and only new items are added.
2. **Interactive review** — a loop shows each element (metadata + collapsible HTML snippet) and prompts you:
   - **`r`** — mark for removal
   - **`k`** — keep (block deletion)
   - **`s`** — skip for now (stays pending)
   - **`q`** — quit and save
3. **Decisions are written to disk immediately after each keypress** — stop, restart kernel, close VSCode — your progress survives.
4. **Apply decisions** — reads the queue, removes only the elements you approved, writes to `cleaned_html/`.

### When to stop and re-run
- Run Step 2a once to generate the queue.
- Run Step 2b as many times as you need — it resumes from where you left off.
- Only run Step 2c when you're satisfied.  
- Step 2d (progress check) can be run any time.

### Reset Progress (if needed)

**Lost track? Want to start over?** Delete the queue and all decisions are wiped. Safe — the original `scraped_html/` is never touched.

In [21]:
# Optional: Reset the queue (wipe all decisions and start fresh)
# Uncomment and run to delete the review_queue.json file
# reset_review_queue(QUEUE_PATH, confirm=True)

In [35]:
# Step 2a — Generate (or resume) the review queue
# Run once. Re-running is safe: existing decisions are NEVER overwritten.
QUEUE_PATH = Path("../data/review_queue_select.json")

queue_path, stats = generate_review_queue(
    html_files,       # list of scraped HTML files
    CANDIDATES,       # your selectors
    QUEUE_PATH,
    soup_cache=soup_cache,
)

print(f"\n✓ Queue saved → {queue_path}")
print(f"  total={stats['total']}  pending={stats['pending']}  remove={stats['remove']}  keep={stats['keep']}")


✓ Queue saved → ..\data\review_queue_select.json
  total=260  pending=260  remove=0  keep=0


In [36]:
# Step 2b — Interactive review loop
# Run (and re-run) this cell to work through pending elements.
# Stop any time with [q] or Ctrl+C — progress is already on disk.
interactive_review(QUEUE_PATH)


  ✓ Progress saved — remove=1  keep=259  pending=0


In [37]:
# Step 2c — Check progress at any time (does not modify anything)
review_progress(QUEUE_PATH)

Progress: [████████████████████] 100%  (260/260)
  🗑  remove  : 1
  🔒 keep    : 259
  ⏳ pending  : 0


,label,remove,keep,pending
0,Select dropdowns (keep 1 option sample),1,259,0


{'total': 260, 'pending': 0, 'remove': 1, 'keep': 259}

In [38]:
# Step 2d — Apply all 'remove' decisions and write cleaned HTML
# Input: scraped_html/ → Output: cleaned_html/ + removal logs (data/removed_log/)
# Safe to re-run: always re-reads the queue from disk.
df_result = apply_queue_decisions(
    src_dir       = Path("../data/cleaned_html_2"),
    queue_path    = QUEUE_PATH,
    out_dir       = Path("../data/cleaned_html_2"),
    removed_log_dir = Path("../data/removed_log_2"),
)

removed_total = df_result["elements_removed"].sum()
print(f"✓ Cleaned {len(df_result)} files — {removed_total} element(s) removed\n")
display(
    df_result[df_result["elements_removed"] > 0]
    .sort_values("elements_removed", ascending=False)
    .reset_index(drop=True)
)

print("\n📋 Per-file removal logs:")
log_dir = Path("../data/removed_log")
if log_dir.exists():
    logs = sorted(log_dir.glob("*_removed.json"))
    print(f"   Found {len(logs)} removal log(s) in {log_dir.resolve()}")
    if logs:
        print("   View any log to see which elements were removed:")
        for log in logs[:3]:  # Show first 3 examples
            print(f"     - {log.name}")
        if len(logs) > 3:
            print(f"     ... and {len(logs) - 3} more")

✓ Removal logs saved → C:\Documents\Kuliah\Skripsi\in-app-navigational-agent\data\removed_log_2

✓ Cleaned 47 files — 1 element(s) removed



,file,elements_removed
0,customer_report_scan_gps.html,1



📋 Per-file removal logs:
   Found 48 removal log(s) in C:\Documents\Kuliah\Skripsi\in-app-navigational-agent\data\removed_log
   View any log to see which elements were removed:
     - customer_attendance_day_off_2_removed.json
     - customer_attendance_day_off_removed.json
     - customer_attendance_employee_schedule_removed.json
     ... and 45 more


---
## Step 2 (Bulk Alternative) — Controlled Removal via OVERRIDE_ALLOW

> **Use this only if you prefer a batch-approval flow over the one-by-one review above.**  
> SAFE selectors (no interactive children) are auto-approved; RISKY ones require an explicit `True` entry in `OVERRIDE_ALLOW`.

**SAFE** selectors (no interactive children) are approved automatically.  
**RISKY** selectors must be explicitly listed in `OVERRIDE_ALLOW` below with a short justification comment explaining why they are safe to remove despite containing interactive descendants.

Selectors absent from `OVERRIDE_ALLOW` *and* flagged RISKY will be **skipped** — their interactive content is preserved.

> After editing `OVERRIDE_ALLOW`, re-run this cell and the removal cell.

In [ ]:
# Step 2a — Declare overrides for RISKY selectors
# Add an entry here only after reviewing the RISKY detail table in Step 1.5.
# True  = "I reviewed it — the interactive children are global chrome, safe to remove"
# False = "Keep it — the interactive content matters for the NKG"

OVERRIDE_ALLOW: dict[str, bool] = {
    # Example:
    # "Mobile nav menu"  : True,   # nav links are duplicated in every page's sidebar → pure chrome
    # "Side Bar Top"     : True,   # user avatar / logout only, not page-specific actions
}

# ── Build approved / blocked sets ────────────────────────────────────────────
safe_labels  = set(df_risk[df_risk["risk"] == "SAFE"]["label"])
risky_labels = set(df_risk[df_risk["risk"] == "RISKY"]["label"])

approved = {
    label: selector
    for label, selector in CANDIDATES.items()
    if label in safe_labels or OVERRIDE_ALLOW.get(label, False)
}
blocked = {
    label: selector
    for label, selector in CANDIDATES.items()
    if label not in approved
}

print("✓ Auto-approved (SAFE):")
for l in sorted(safe_labels & approved.keys()):
    print(f"    {l}")

overridden = [l for l in risky_labels if OVERRIDE_ALLOW.get(l, False)]
if overridden:
    print("\n✓ Override-approved (RISKY but explicitly allowed):")
    for l in sorted(overridden):
        print(f"    {l}")

if blocked:
    print("\n✗ Blocked (RISKY, no override — will NOT be removed):")
    for l in sorted(blocked):
        print(f"    {l}")
else:
    print("\nAll selectors approved — nothing blocked.")

In [ ]:
# Step 2b — Remove approved elements & save cleaned HTML
# Only selectors in `approved` (SAFE + explicitly overridden RISKY) are stripped.
# Input: scraped_html/  →  Output: cleaned_html/
src_dir = Path("../data/scraped_html")
out_dir = Path("../data/cleaned_html")
src_files = sorted(src_dir.glob("*.html"))

if not approved:
    print("Nothing to remove — `approved` dict is empty. Check Step 2a.")
else:
    df_summary = remove_elements_and_save(src_files, approved, out_dir)
    print(f"Cleaned {len(src_files)} files  →  {out_dir.resolve()}\n")
    display(df_summary[["file", "total_removed"]].sort_values("total_removed", ascending=False))
    
    if blocked:
        print(f"\n⚠  Skipped {len(blocked)} RISKY selector(s) — add them to OVERRIDE_ALLOW in Step 2a if you want to remove them:")
        for l in sorted(blocked):
            print(f"    {l}")

Cleaned 47 files  →  C:\Documents\Kuliah\Skripsi\in-app-navigational-agent\data\cleaned_html

                                          file  total_removed
                                      107.html              2
    customer_attendance_employee_schedule.html              2
                customer_attendance_leave.html              2
      customer_attendance_leave_allowance.html              2
             customer_attendance_overtime.html              2
             customer_attendance_schedule.html              2
               customer_attendance_status.html              2
                            customer_cart.html              2
                       customer_changelog.html              2
                       customer_dashboard.html              2
                          customer_device.html              2
                  customer_device_api_sdk.html              2
              customer_device_push_server.html              2
                        customer_emplo

## Step 3 — Lint & Format HTML

Lint and prettify all HTML files in a specified folder.  
Formatted files are written back to the same folder (or to a new output folder if preferred).

In [11]:
# Step 3 — Lint & Format HTML
folder_to_format = Path("../data/cleaned_html_2")  # Change this path as needed
df_format = lint_and_format_html(folder_to_format)
print(f"Linted & formatted {len(df_format)} files in {folder_to_format.resolve()}\n")
print(df_format.to_string(index=False))

Linted & formatted 47 files in C:\Documents\Kuliah\Skripsi\in-app-navigational-agent\data\cleaned_html_2

                                          file  original_size  formatted_size
              customer_attendance_day_off.html          45475           45475
    customer_attendance_employee_schedule.html          41344           41344
                customer_attendance_leave.html         177437          177437
      customer_attendance_leave_allowance.html          21576           21576
             customer_attendance_overtime.html          75856           75856
             customer_attendance_schedule.html          28719           28719
               customer_attendance_status.html          77887           77887
                            customer_cart.html         324150          324150
                       customer_changelog.html          34398           34398
                       customer_dashboard.html          52546           52546
                          customer_d

---

## 📋 Removal Logs

After Step 2d, every file that had elements removed gets a JSON log in `data/removed_log/{filename}_removed.json`.

Each log contains:
- **total_removed**: count of removed elements
- **removed_elements**: array of each removed element with:
  - `label`: which selector it matched ("Mobile nav menu", etc.)
  - `tag`, `id`, `classes`, `text_preview`: HTML metadata
  - `html`: the full prettified element (for auditing)

**Use these logs to:**
- Verify you removed the right things
- Audit what happened if something breaks in the NKG
- Refer back if you need to undo a decision

Load a log in Python:
```python
import json
with open("../data/removed_log/customer_dashboard_removed.json") as f:
    log = json.load(f)
print(f"Removed {log['total_removed']} element(s)")
for el in log['removed_elements']:
    print(f"  - {el['label']} ({el['tag']})")
```